# Claude Code + AgentCore Gateway (Native OAuth)

This notebook configures **Claude Code** to connect to the AgentCore Gateway using its **native MCP OAuth flow**. When Claude Code connects to the Gateway, it:

1. Receives a `401` with OAuth metadata pointing to the Okta authorization server
2. Opens a **browser tab** for Okta login (Authorization Code + PKCE)
3. Receives the JWT and caches it for subsequent requests
4. Gateway enforces **Cedar policies** based on the authenticated user's identity

No token management code needed — Claude Code handles the entire OAuth lifecycle.

```
Claude Code
  │
  │ MCP StreamableHTTP → 401 → OAuth discovery
  │
  │ Browser popup → Okta login (Authorization Code + PKCE)
  │
  │ JWT Bearer token (cached automatically)
  ▼
AgentCore Gateway (CUSTOM_JWT authorizer)
  ├── Validates JWT (signature, audience, client_id)
  ├── Cedar Policy Engine (ENFORCE mode)
  │     principal: AgentCore::OAuthUser::"<JWT sub>"
  │
  ├───────────────┐
  ▼               ▼
Lambda          Lambda
(Weather)       (Finance)
```

## Prerequisites

- **`01_setup.ipynb`** completed (Gateway, Lambda targets, Cedar policies deployed)
- **`gateway_config.json`** exists with deployed resource IDs
- **Claude Code** installed (`npm install -g @anthropic-ai/claude-code`)
- **Okta API token** in `.env` (for creating the SPA app)

## Cell 1: Load Configuration

Load the Gateway config from `01_setup.ipynb` and Okta credentials from `.env`.

In [ ]:
import os
import json
import requests
import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Load Gateway config from 01_setup ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_ID = config["gateway_id"]
GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_ISSUER = config["okta_issuer"]
OKTA_ORIGINAL_CLIENT_ID = config["okta_client_id"]  # Web app (confidential client)
AWS_REGION = config["aws_region"]

# --- Okta API token (for creating SPA app) ---
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]

print(f"Gateway ID:     {GATEWAY_ID}")
print(f"Gateway URL:    {GATEWAY_URL}")
print(f"Okta Domain:    {OKTA_DOMAIN}")
print(f"Okta Issuer:    {OKTA_ISSUER}")
print(f"Original Client: {OKTA_ORIGINAL_CLIENT_ID} (Web app)")
print(f"AWS Region:     {AWS_REGION}")
print(f"\nOAuth Discovery (Gateway -> Okta):")

# Verify the Gateway returns OAuth metadata
resp = requests.get(f"{GATEWAY_URL.replace('/mcp', '')}/.well-known/oauth-protected-resource", timeout=10)
print(f"  Protected Resource Metadata: {resp.json()}")

## Cell 2: Create Okta SPA Application

Claude Code's OAuth flow uses **Authorization Code + PKCE** — a public client flow that doesn't require a client secret. The existing Okta app from `01_setup` is a confidential "Web" app (requires `client_secret`), so we create a separate **SPA (Single Page Application)** type app.

Key differences:
| | Web App (02_demo) | SPA App (Claude Code) |
|--|---|---|
| **Type** | `web` (confidential) | `browser` (public) |
| **Auth method** | `client_secret_basic` | `none` (PKCE only) |
| **Grant type** | `password` (ROPC) | `authorization_code` |
| **Token flow** | Programmatic (no browser) | Browser popup |

In [ ]:
OKTA_HEADERS = {
    "Authorization": f"SSWS {OKTA_API_TOKEN}",
    "Content-Type": "application/json",
}

CALLBACK_PORT = 8400  # Claude Code's OAuth callback port

# --- Create SPA app for Claude Code ---
spa_payload = {
    "name": "oidc_client",
    "label": "AgentCore Gateway - Claude Code (SPA)",
    "signOnMode": "OPENID_CONNECT",
    "settings": {
        "oauthClient": {
            "application_type": "browser",
            "grant_types": ["authorization_code"],
            "response_types": ["code"],
            "redirect_uris": [f"http://localhost:{CALLBACK_PORT}/callback"],
            "post_logout_redirect_uris": [f"http://localhost:{CALLBACK_PORT}"],
        }
    },
    "credentials": {
        "oauthClient": {
            "token_endpoint_auth_method": "none",
        }
    },
}

resp = requests.post(
    f"https://{OKTA_DOMAIN}/api/v1/apps",
    headers=OKTA_HEADERS,
    json=spa_payload,
)

if resp.status_code == 200:
    spa_app = resp.json()
    SPA_CLIENT_ID = spa_app["credentials"]["oauthClient"]["client_id"]
    print(f"Created SPA app: {spa_app['label']}")
    print(f"  Client ID: {SPA_CLIENT_ID}")
    print(f"  Auth method: {spa_app['credentials']['oauthClient']['token_endpoint_auth_method']}")
    print(f"  Redirect URI: http://localhost:{CALLBACK_PORT}/callback")
else:
    print(f"Error creating app: {resp.status_code}")
    print(resp.json())
    raise SystemExit("Fix the error above before continuing")

# --- Assign app to Everyone group ---
everyone_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/groups?q=Everyone&limit=1",
    headers=OKTA_HEADERS,
)
everyone_group_id = everyone_resp.json()[0]["id"]

assign_resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{spa_app['id']}/groups/{everyone_group_id}",
    headers=OKTA_HEADERS,
)
if assign_resp.status_code == 200:
    print(f"\nAssigned to Everyone group")
else:
    print(f"\nWarning: Could not assign to Everyone group: {assign_resp.status_code}")

print(f"\nSPA Client ID: {SPA_CLIENT_ID}")

## Cell 3: Configure Okta Default Scopes

Claude Code follows the MCP OAuth spec (RFC 9728) and sends a `resource` parameter in the authorization request. Okta requires either a `scope` parameter or default scopes configured on the authorization server. Since Claude Code doesn't send `scope` by default, we must set default scopes on the Okta authorization server.

Without this, Okta returns: *"The authorization server resource does not have any configured default scopes, 'scope' must be provided."*

In [ ]:
# --- Set custom scopes as default ---
# System scopes (openid, profile, etc.) cannot be set as default via API,
# but custom scopes like "groups" can. We also publish "groups" so it appears
# in the OpenID discovery document.

scopes_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default/scopes",
    headers=OKTA_HEADERS,
)
scopes = scopes_resp.json()

for scope in scopes:
    if scope["name"] == "groups" and not scope.get("default"):
        # Set groups scope as default
        scope_update = {k: v for k, v in scope.items() if k != "_links"}
        scope_update["default"] = True
        scope_update["metadataPublish"] = "ALL_CLIENTS"

        update_resp = requests.put(
            f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default/scopes/{scope['id']}",
            headers=OKTA_HEADERS,
            json=scope_update,
        )
        if update_resp.status_code == 200:
            print(f"Set '{scope['name']}' as default scope")
        else:
            print(f"Warning: Could not update '{scope['name']}': {update_resp.text[:200]}")
    elif scope["name"] == "groups" and scope.get("default"):
        print(f"'{scope['name']}' is already a default scope")

# Show final scope state
print("\nAuthorization server scopes:")
for scope in scopes:
    default = "DEFAULT" if scope.get("default") else ""
    system = "system" if scope.get("system") else "custom"
    print(f"  {scope['name']:20s} {system:8s} {default}")

## Cell 4: Update Gateway `allowedClients`

The AgentCore Gateway validates the JWT `client_id` claim against its `allowedClients` list. We need to add the new SPA client ID alongside the original web app client ID.

In [ ]:
import time

agentcore = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

# --- Get current Gateway config ---
gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]
print(f"Current allowedClients: {current_clients}")

# --- Add SPA client ID if not already present ---
if SPA_CLIENT_ID in current_clients:
    print(f"\nSPA client ID already in allowedClients — no update needed")
else:
    new_clients = current_clients + [SPA_CLIENT_ID]
    response = agentcore.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Updated allowedClients: {new_clients}")
    print(f"Gateway status: {response['status']}")

    # Wait for READY
    for attempt in range(12):
        gw = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
        if gw["status"] == "READY":
            print("Gateway READY")
            break
        time.sleep(5)
        print(f"  Status: {gw['status']} ({(attempt + 1) * 5}s)")

## Cell 5: Configure Claude Code MCP Server

This adds the AgentCore Gateway as an MCP server in Claude Code's project config.

**Key config fields:**
- `type: http` — MCP StreamableHTTP transport
- `url` — Gateway's MCP endpoint
- `oauth.clientId` — Okta SPA app client ID (public client, no secret)
- `oauth.callbackPort` — Port for the OAuth redirect callback
- `oauth.scope` — OAuth scopes to request (`openid` for OIDC, `groups` for Cedar identity)
- `oauth.authorizationUrl` / `oauth.tokenUrl` — Explicit Okta endpoints (required because Claude Code's MCP OAuth discovery sends a `resource` parameter that Okta doesn't handle without explicit scopes)

In [ ]:
import subprocess

mcp_config = {
    "type": "http",
    "url": GATEWAY_URL,
    "oauth": {
        "clientId": SPA_CLIENT_ID,
        "callbackPort": CALLBACK_PORT,
        "scope": "openid groups",
        "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
        "tokenUrl": f"{OKTA_ISSUER}/v1/token",
    },
}

print("MCP Server config:")
print(json.dumps(mcp_config, indent=2))

# --- Add to Claude Code (project scope) ---
result = subprocess.run(
    ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(mcp_config)],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print(f"\nAdded MCP server to Claude Code")
    print(result.stdout.strip())
else:
    # May already exist — try removing and re-adding
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(mcp_config)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"\nUpdated MCP server in Claude Code")
        print(result.stdout.strip())
    else:
        print(f"\nError: {result.stderr}")

# --- WORKAROUND: Claude Code may not persist the 'scope' field ---
# Patch .claude.json directly to ensure scope and URLs are present
claude_json_path = os.path.expanduser("~/.claude.json")
with open(claude_json_path) as f:
    claude_config = json.load(f)

project_key = os.getcwd()
if "projects" in claude_config and project_key in claude_config["projects"]:
    servers = claude_config["projects"][project_key].get("mcpServers", {})
    if "agentcore-gateway" in servers:
        servers["agentcore-gateway"]["oauth"]["scope"] = "openid groups"
        servers["agentcore-gateway"]["oauth"]["authorizationUrl"] = f"{OKTA_ISSUER}/v1/authorize"
        servers["agentcore-gateway"]["oauth"]["tokenUrl"] = f"{OKTA_ISSUER}/v1/token"
        with open(claude_json_path, "w") as f:
            json.dump(claude_config, f, indent=2)
        print("\nPatched .claude.json with scope and OAuth URLs")

print(f"\n--- Summary ---")
print(f"Gateway:        {GATEWAY_URL}")
print(f"Okta SPA Client: {SPA_CLIENT_ID}")
print(f"Callback Port:  {CALLBACK_PORT}")
print(f"Scopes:         openid groups")

## Cell 6: Save Config

Save the SPA client ID to `gateway_config.json` for reference.

In [ ]:
config["okta_spa_client_id"] = SPA_CLIENT_ID
config["claude_code_callback_port"] = CALLBACK_PORT

with open("gateway_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved SPA client ID to gateway_config.json")
print(f"\n  okta_spa_client_id: {SPA_CLIENT_ID}")
print(f"  claude_code_callback_port: {CALLBACK_PORT}")

## Test It

Open a **new Claude Code session** in this project directory:

```bash
cd /path/to/demo-centralized-mcp-server
claude
```

On startup, Claude Code will connect to the `agentcore-gateway` MCP server:

1. A **browser tab opens** with the Okta login page
2. Log in as **Alice** (`analysts` group) or **Bob** (`finance-admins` group)
3. After authentication, Claude Code displays: *"Authentication successful. Connected to agentcore-gateway."*
4. Run `/mcp` to see available tools

### Expected Results

| Login as | Tools visible | Reason |
|----------|--------------|--------|
| **Alice** | `WeatherTools___get_weather` | Cedar permits all `OAuthUser` principals for WeatherTools |
| **Bob** | `WeatherTools___get_weather` + `FinanceTools___get_revenue_data` | Cedar permits Bob specifically for FinanceTools |

### Try It

```
> What's the weather in Sydney?
> Show me the quarterly revenue data.
```

When logged in as Alice, the revenue query will fail (Cedar DENY). When logged in as Bob, both queries succeed.

### Switching Users

To test as a different user, clear the cached OAuth token and restart:

```bash
# Remove cached OAuth tokens
rm -rf ~/.claude/oauth-tokens/
# Restart Claude Code
claude
```

## Cleanup

Remove the SPA app from Okta and revert the Gateway's `allowedClients`. This does **not** remove the Gateway or Lambda targets (use `01_setup.ipynb` cleanup for that).

In [ ]:
# --- Cleanup: revert Gateway and delete SPA app ---

import json
import boto3
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

with open("gateway_config.json") as f:
    cfg = json.load(f)

SPA_CLIENT_ID = cfg.get("okta_spa_client_id", "")
OKTA_DOMAIN = cfg["okta_domain"]
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]
OKTA_HEADERS = {"Authorization": f"SSWS {OKTA_API_TOKEN}", "Content-Type": "application/json"}

# 1. Revert Gateway allowedClients to original (web app only)
agentcore = boto3.client("bedrock-agentcore-control", region_name=cfg["aws_region"])
gw = agentcore.get_gateway(gatewayIdentifier=cfg["gateway_id"])
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]

if SPA_CLIENT_ID and SPA_CLIENT_ID in current_clients:
    new_clients = [c for c in current_clients if c != SPA_CLIENT_ID]
    agentcore.update_gateway(
        gatewayIdentifier=cfg["gateway_id"],
        name=gw["name"],
        roleArn=gw["roleArn"],
        protocolType=gw["protocolType"],
        authorizerType=gw["authorizerType"],
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": gw["authorizerConfiguration"]["customJWTAuthorizer"]["discoveryUrl"],
                "allowedAudience": gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedAudience"],
                "allowedClients": new_clients,
            }
        },
        policyEngineConfiguration=gw.get("policyEngineConfiguration", {}),
        exceptionLevel="DEBUG",
    )
    print(f"Reverted Gateway allowedClients to: {new_clients}")

# 2. Delete Okta SPA app
if SPA_CLIENT_ID:
    apps_resp = requests.get(
        f"https://{OKTA_DOMAIN}/api/v1/apps?q=AgentCore Gateway - Claude Code",
        headers=OKTA_HEADERS,
    )
    for app in apps_resp.json():
        if app.get("credentials", {}).get("oauthClient", {}).get("client_id") == SPA_CLIENT_ID:
            # Deactivate then delete
            requests.post(f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}/lifecycle/deactivate", headers=OKTA_HEADERS)
            requests.delete(f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}", headers=OKTA_HEADERS)
            print(f"Deleted Okta SPA app: {app['label']} ({SPA_CLIENT_ID})")
            break

# 3. Remove from Claude Code
import subprocess
subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
print("Removed agentcore-gateway from Claude Code")

# 4. Clean up gateway_config.json
cfg.pop("okta_spa_client_id", None)
cfg.pop("claude_code_callback_port", None)
with open("gateway_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("\nCleanup complete")